In [1]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

torch.cuda.empty_cache()

model_name = "Qwen/Qwen3-0.6B"
model_save_dir = "model/downloaded"
os.makedirs(model_save_dir, exist_ok=True)

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="cuda",
    cache_dir=model_save_dir
)


In [ ]:
from pruning.utils import find_layers, check_sparsity
from data.datasets import get_loaders
def prepare_calibration_input(model, dataloader, device):
    # import pdb; pdb.set_trace()
    use_cache = model.config.use_cache
    model.config.use_cache = False
    layers = model.model.layers

    # dev = model.hf_device_map["model.embed_tokens"]
    if "model.embed_tokens" in model.hf_device_map:
        device = model.hf_device_map["model.embed_tokens"]

    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros((128, model.seqlen, model.config.hidden_size), dtype=dtype, device=device)
    inps.requires_grad = False
    cache = {'i': 0, 'attention_mask': None, "position_ids": None}

    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp
            cache['i'] += 1
            cache['attention_mask'] = kwargs['attention_mask']
            cache['position_ids'] = kwargs['position_ids']
            raise ValueError
    
    layers[0] = Catcher(layers[0])
    for batch in dataloader:
        try:
            model(batch[0].to(device))
        except ValueError:
            pass 
    layers[0] = layers[0].module

    outs = torch.zeros_like(inps)
    attention_mask = cache['attention_mask']
    # import pdb; pdb.set_trace()
    # attention_mask = cache['attention_mask'] or torch.ones(inps.shape[:2], device=inps.device, dtype=torch.bool)
    position_ids = cache['position_ids']
    model.config.use_cache = use_cache

    return inps, outs, attention_mask, position_ids 


In [ ]:
inps, outs, attention_mask, position_ids = prepare_calibration_input(model, None, "cuda")